# UCS761 - Deep Learning Lab 5
### From Linear Estimation to Learning Representations

Linear models from the previous labs can only fit a straight line or a flat surface.
If the data curves, they just cannot capture that no matter how long you train.

This lab covers:
1. Why linear models fail on non-linear data
2. How a hidden layer lets the model bend its output
3. Manual backpropagation through a two-layer network
4. Comparing linear vs non-linear model on the same data

No sklearn, no torch. Just numpy.

In [1]:
import numpy as np


## Step 1: Create the Dataset

We use log(X+1) + noise as the target.

Good choice because:
It grows fast near x=0 then slows down near x=10.
No straight line can capture this since the slope keeps changing.
The range 0 to 10 shows both the steep part and the flat part.

In [2]:
np.random.seed(42)

X = np.linspace(0, 10, 50).reshape(-1, 1)
noise = np.random.normal(0, 0.2, size=(50, 1))
y = np.log(X + 1) + noise

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X range:", X.min(), "to", X.max())
print("y range:", round(y.min(), 3), "to", round(y.max(), 3))


X shape: (50, 1)
y shape: (50, 1)
X range: 0.0 to 10.0
y range: 0.099 to 2.572


## Step 2: Initialize Weights

Architecture: 1 input, 3 hidden neurons, 1 output

Two weight matrices:
W1 (1x3): connects input to hidden neurons
W2 (3x1): connects hidden neurons to output

Weights randomly initialized in [-1, 1]. Biases start at 0.

Why this matters: if all weights were the same, all neurons would learn the exact same thing. Random initialization breaks that symmetry. Weights control the shape of the function, biases shift it up or down.

In [3]:
W1 = np.random.uniform(-1, 1, size=(1, 3))
b1 = np.zeros((1, 3))
W2 = np.random.uniform(-1, 1, size=(3, 1))
b2 = np.zeros((1, 1))

print("W1:", W1.shape, "  W2:", W2.shape)


W1: (1, 3)   W2: (3, 1)


## Step 3: Activation Function

Tanh is used for the hidden layer.

Why tanh and not sigmoid: tanh is centered at 0 with output range -1 to 1, which works better for hidden layers. Sigmoid is always positive which can cause weights to update mostly in one direction.

Derivative: 1 - tanh(z)^2

This derivative is used in backpropagation to control how much of the error signal gets passed back to earlier layers.

In [4]:
def tanh(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z) ** 2


## Step 4: Forward Pass

Three steps:
1. z1 = X @ W1 + b1 (linear combination, input to hidden)
2. h = tanh(z1) (apply activation, adds non-linearity)
3. y_hat = h @ W2 + b2 (output layer, no activation for regression)

We store z1 and h because we need them during backpropagation.

In [5]:
def forward(X, W1, b1, W2, b2):
    z1    = X @ W1 + b1
    h     = tanh(z1)
    y_hat = h @ W2 + b2
    return z1, h, y_hat


## Step 5: MSE Loss

Regression problem so we use MSE.

In [6]:
def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)


## Step 6: Backward Pass

Chain rule applied layer by layer, going backwards from the output.

Output layer:
dL/dy_hat = 2*(y_hat - y)/N
dL/dW2 = h^T @ dL/dy_hat
dL/db2 = sum of dL/dy_hat

Hidden layer:
dL/dh = dL/dy_hat @ W2^T (propagate error back through W2)
dL/dz1 = dL/dh * tanh_deriv(z1) (apply activation slope here)
dL/dW1 = X^T @ dL/dz1
dL/db1 = sum of dL/dz1

The activation slope at the hidden layer is the key part. If tanh is saturated (outputs near plus or minus 1), the derivative is near 0 and the gradient dies. Weights stop updating.

In [7]:
def backward(X, y, z1, h, y_hat, W2):
    n = len(y)

    dout   = 2 * (y_hat - y) / n

    dW2    = h.T @ dout
    db2    = np.sum(dout, axis=0, keepdims=True)

    dh     = dout @ W2.T
    dz1    = dh * tanh_deriv(z1)

    dW1    = X.T @ dz1
    db1    = np.sum(dz1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2


## Step 7: Training Loop

In [8]:
lr     = 0.01
losses = []

for epoch in range(5000):
    z1, h, y_hat = forward(X, W1, b1, W2, b2)
    l            = mse(y, y_hat)
    losses.append(l)

    dW1, db1, dW2, db2 = backward(X, y, z1, h, y_hat, W2)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1:5d}  Loss: {l:.4f}")


Epoch  1000  Loss: 0.0318
Epoch  2000  Loss: 0.0303
Epoch  3000  Loss: 0.0299
Epoch  4000  Loss: 0.0297
Epoch  5000  Loss: 0.0295


## Step 8: Compare with a Linear Model

If a straight line was fit to the same data, how bad would it be?

In [9]:
cov_matrix = np.cov(X.ravel(), y.ravel())
w_lin = cov_matrix[0, 1] / np.var(X)
b_lin = y.mean() - w_lin * X.mean()

y_lin = X * w_lin + b_lin

lin_loss = mse(y, y_lin)

_, _, y_nn = forward(X, W1, b1, W2, b2)
nn_loss = mse(y, y_nn)

print(f"Linear model MSE:     {lin_loss:.4f}")
print(f"Two-layer NN MSE:     {nn_loss:.4f}")
print(f"NN is {(lin_loss - nn_loss) / lin_loss * 100:.1f}% better")


Linear model MSE:     0.0512
Two-layer NN MSE:     0.0295
NN is 42.3% better


In [10]:
print("Sample predictions (first 8 points):")
print(f"{'x':>6} | {'actual y':>10} | {'linear':>10} | {'NN':>10}")
for i in range(8):
    print(f"  {X[i,0]:4.1f}  |   {y[i,0]:7.3f}   |   {y_lin[i,0]:7.3f}   |   {y_nn[i,0]:7.3f}")


Sample predictions (first 8 points):
     x |   actual y |     linear |         NN
   0.0  |     0.099   |     0.601   |     0.110
   0.2  |     0.158   |     0.642   |     0.285
   0.4  |     0.472   |     0.682   |     0.442
   0.6  |     0.782   |     0.722   |     0.577
   0.8  |     0.550   |     0.762   |     0.688
   1.0  |     0.656   |     0.802   |     0.778
   1.2  |     1.115   |     0.842   |     0.852
   1.4  |     1.041   |     0.882   |     0.915


## Observations

**Why the linear model fails:**
log(x+1) is a curve that rises quickly near 0 and then levels off. A straight line has a fixed slope. It cannot match steep here, flat there. It will either overestimate the left side or underestimate the right side, never both at once.

**What the hidden layer does:**
Each hidden neuron learns a different transformation of the input. Combined, multiple neurons can approximate complex curved functions. This is essentially function approximation.

**Activation slope and gradient flow:**
If tanh is saturated, slope is near 0, gradient is near 0, no learning happens. This is why initialization matters. We want activations in the active zone where slope is meaningful.

**Why store z1 and h during forward pass:**
These are needed for backprop. Without storing intermediate values you would have to recompute everything, which wastes time and can cause issues if random operations are involved.